# XLeRobot Digital Twin — Deployment

**Course:** RBB2013 Digital Twin (May 2026)  
**Repository:** https://github.com/ManHazz/RBB2013-Digital-Twin

**Team members:**

| Name | Student ID |
|------|------------|
| Muhammad Aiman bin Ahmad Hazimin | 22011708 |
| Hazieq Danial bin Roshihan Annuar | 24006633 |
| Muhammad Raziq bin Sufian | 24006626 |
| Ibrohim bin Ahmad Jaafar Sadzik | 24006396 |
| Ariq Danish bin Nor Razak | 24006796 |

**Rubric:** Deployment (5%)

> *"Partition your digital twin design into well defined microservices and specify exactly the function and operation of each and all microservices. Specify the contract information which states the route, port, protocol format and data format exchange between every pair of microservices and when that communication is initiated and concluded. Build containers for each microservices, demonstrate successful deployment and scaling of microservices. Prove that the data captured and stored by the microservices is persistent. Develop and demonstrate a complete test suite to test and microservices, interface and overall digital twin system operation."*


## 1. Well-defined microservices — function and operation

| # | Service | Was | Container | Port(s) | Language / framework | Responsibility |
|---|---------|-----|-----------|---------|----------------------|----------------|
| 1 | `nl-command` | `llm_controller.py` | ✅ | 8010 | Python / FastAPI | Take user text, call Ollama LLM, return `TargetPose` |
| 2 | `motion-planner` | `robot_ik.py` | ✅ | 8020 (internal) | Python / FastAPI | IK solve + reachability + collision check |
| 3 | `planner-lb` | new | ✅ | 8020 (public) | nginx | Round-robin `motion-planner` replicas for horizontal scale |
| 4 | `dispatcher` | interpolator | ✅ | 8030, 5556 | Python / FastAPI + pyzmq | 30-fps interpolation, ZMQ PUSH frames to sim, HTTP-trigger actuation |
| 5 | `sim-bridge` | `extension.py` | ❌ (host, GPU) | 5556 PULL, 5557 PUB | Python / Omniverse Kit | Apply joints in USD scene, publish live `SimState` |
| 6 | `actuation` | MQTT publisher | ✅ | 8040 | Python / FastAPI + paho-mqtt | On dispatcher's HTTP trigger, publish `xlerobot/cmd` |
| 7 | `telemetry` | new | ✅ | daemon | Python / pyzmq + psycopg + redis | SUB sim state → write TimescaleDB row + Redis latest |
| — | `ollama` | infra | 🟡 host | 11434 | LLM runtime | LLM inference (Qwen2.5 3B) |
| — | `timescaledb` | infra | ✅ | 5432 | TimescaleDB 2.x on Postgres 15 | Time-series persistence |
| — | `redis` | infra | ✅ | 6379 | Redis 7 (AOF+RDB) | Latest-state cache |
| — | `mosquitto` | infra | ✅ | 1883 | Eclipse Mosquitto 2 | MQTT broker |
| — | `grafana` | infra | ✅ | 3000 | Grafana 10.4 | Time-series visualization |

### Why sim-bridge lives on the host

`sim-bridge` is the Omniverse Kit extension `digitaltwin.xlerobot_extension`. It must run **on the host** because Omniverse Kit needs GPU-accelerated Vulkan, direct X server access, and Kit's own bundled Python. Containerizing would need GPU passthrough + X11 forwarding for negligible benefit. The boundary is deliberate: sim-bridge lives on the host and talks to the compose network via `host.docker.internal:5556/5557`. All other services are containerized.

## 2. Interface contract — route, port, protocol, data format, initiation, conclusion

For every pair of communicating microservices. Full per-pair contracts with payload examples in `contracts/interface-contracts.md`.

| From → To | Route/Topic | Port | Protocol | Data format | Initiated | Concluded |
|-----------|-------------|------|----------|-------------|-----------|-----------|
| client → nl-command | `POST /command` | 8010 | HTTP/JSON | `{text}` → `{x,y,z}` | user submits text | pose returned |
| nl-command → ollama | `POST /api/generate` | 11434 | HTTP/JSON | prompt → completion | on each command | completion returned |
| nl-command → motion-planner | `POST /plan` | 8020 | HTTP/JSON | `{target}` → `{joints[6], reachable, collision_free}` | pose resolved | plan returned |
| motion-planner → dispatcher | `POST /dispatch` | 8030 | HTTP/JSON | `{joints[6]}` → `{accepted}` | plan valid | ack |
| dispatcher → sim-bridge | joint frames | 5556 | ZMQ PUSH/PULL | `{joints[6], frame_id}` | dispatch accepted | last (30th) frame sent |
| sim-bridge → telemetry | sim state | 5557 | ZMQ PUB/SUB | `{joints[6], ee_pose, target, obstacles, ts}` | every sim tick (10 Hz) | run ends |
| telemetry → timescaledb | INSERT | 5432 | SQL | hypertable row `robot_state` | on each state msg | commit |
| telemetry → redis | SET | 6379 | RESP | key `state:latest`, JSON value | on each state msg | overwritten by next |
| dispatcher → actuation | `POST /actuate` | 8040 | HTTP/JSON | `{joints[6]}` → `{published, topic}` | after last sim frame sent | MQTT rc = 0 |
| actuation → mosquitto | `xlerobot/cmd` | 1883 | MQTT/JSON | `{joints[6]}` | run validated in sim | broker ack |
| grafana → timescaledb | SELECT | 5432 | SQL | time-series read | dashboard refresh (5 s) | rows returned |

**Single contract source of truth:** `services/shared/schemas.py` — pydantic v2 models used by every service. Frozen v1.0 in sprint 1. Only the tech lead edits this file. Contract violation at any hop → HTTP 422.

## 3. Containers — one Dockerfile per service

All 5 app services + 4 infra services (Timescale, Redis, Mosquitto, Grafana) run under `docker compose`. Compose file: `infra/docker-compose.yml`.

### 3.1 Successful deployment — `docker compose ps`

![Compose PS](./screenshots/docker_compose_ps.png)

*All 10 containers Up. Timescale reports `healthy` from its `pg_isready` healthcheck.*

In [ ]:
# Bring the full stack up (from the repo root)

!docker compose -f infra/docker-compose.yml up -d
!docker compose -f infra/docker-compose.yml ps

## 4. End-to-end demonstration — command → arm moves → data flows

### 4.1 Terminal — chain the full pipeline

![Command flow](./screenshots/command_flow.png)

1. User's natural-language command `"pick up the ball"`.
2. LLM parsed → `TargetPose(x=40, y=13.75, z=0)`.
3. Motion-planner IK → 6 joint angles, `reachable=true, collision_free=true`.
4. Dispatcher accepts → streams 30 interpolated frames → triggers actuation MQTT.

### 4.2 Omniverse — the digital twin arm reaches the target

![Omniverse arm reaching](./screenshots/omniverse_arm_reaching.png)

*Gripper reaches the red target ball, all under real IK + collision check.*

### 4.3 Grafana — live state visualization

![Grafana healthy](./screenshots/grafana_dashboard_healthy.png)

*The **Pipeline health** panel confirms the deployment is delivering data end-to-end (HEALTHY = ≥300 state messages/min).*

## 5. Scaling — horizontal replicas of stateless microservice

`motion-planner` is fully stateless (IK is pure math per request). Scaled horizontally behind an nginx load balancer using Docker's built-in DNS to reach all replicas.

```bash
docker compose -f infra/docker-compose.yml up -d --scale motion-planner=3
```

Round-robin load balancer config in `infra/nginx/nginx.conf`. Full scaling demo procedure + burst-test measurements in `docs/SCALING_PROOF.md`.

**Why motion-planner?** Because it has no per-request state. `dispatcher` holds ZMQ socket state (connected sim-bridge peer) so it cannot trivially replicate.

## 6. Persistence — data (TimescaleDB) + state (Redis) both survive restart

The rubric splits persistence into two halves: **data storage** (historical) and **state storage** (latest). Both proven independently.

### 6.1 Before restart

**TimescaleDB row count:**

![Persistence before Timescale](./screenshots/persistence_before_timescale.png)

**Redis latest state:**

![Persistence before Redis](./screenshots/persistence_before_redis.png)

### 6.2 Restart both persistence containers

```bash
docker compose -f infra/docker-compose.yml restart timescaledb redis
```

### 6.3 After restart — data and state both survived

**TimescaleDB row count:**

![Persistence after Timescale](./screenshots/persistence_after_timescale.png)

*46,570 → 47,100 rows. Not only did old data survive, new data continued arriving during the restart window (telemetry auto-reconnects).*

**Redis latest state:**

![Persistence after Redis](./screenshots/persistence_after_redis.png)

*Byte-identical `state:latest` JSON. Named Docker volumes on both `timescaledb-data` (Postgres data) and `redis-data` (AOF + RDB) guarantee durability.*

### 6.4 Design decisions

- **TimescaleDB** — hypertable on `robot_state(ts, ...)` for cheap time-range queries as history grows. Named volume on `/var/lib/postgresql/data`. Survives `docker compose restart` and `docker compose down` (not `down -v`).
- **Redis** — `--appendonly yes --save 60 1` — AOF for durability + periodic RDB snapshots. Named volume on `/data`. `state:latest` survives restart.

## 7. Complete test suite — microservices, interface, system

Three tiers, all CI-gated.

```
tests/
├── unit/         — per-service pure logic, no infra, fast
├── integration/  — service pairs against REAL infra (testcontainers spins Timescale, Redis, mosquitto)
├── system/       — full stack via docker compose, end-to-end assertion (command → row in Timescale)
└── regression/   — golden IK cases, run on every CI push, fails if math drifts
```

### 7.1 Unit tests — per microservice, pass AND fail cases

| Test file | Service | Pass case | Fail case |
|-----------|---------|-----------|-----------|
| `test_nl_command.py` | nl-command | happy-path pose parse | empty text → 422, garbled LLM → 422 |
| `test_motion_planner.py` | motion-planner | IK round-trip < 1e-3 | unreachable → `reachable=false`, colliding → `collision_free=false` |
| `test_dispatcher.py` | dispatcher | 30 frames sent, first/last frame correct | — |
| `test_actuation.py` | actuation | correct MQTT topic + payload | wrong joint count → 422 |
| `test_telemetry.py` | telemetry | Timescale insert + Redis SET verified | — |

### 7.2 Integration tests — every service pair, real infra

| Test file | Pair tested | Real infra |
|-----------|-------------|-----------|
| `test_nl_to_planner.py` | nl-command → motion-planner | live TestClient chain, Ollama stubbed |
| `test_planner_to_dispatcher.py` | motion-planner → dispatcher | live TestClient chain, ZMQ mocked |
| `test_actuation_mqtt.py` | actuation → mosquitto | testcontainers spins real `eclipse-mosquitto:2` |
| `test_sim_to_telemetry.py` | sim state → Timescale + Redis | testcontainers spins real Timescale + Redis |

### 7.3 System test — full stack end-to-end

`tests/system/test_end_to_end.py` — brings up the compose stack, POSTs `/command`, asserts a `robot_state` row lands in TimescaleDB within 10 s.

### 7.4 Regression suite — CI-gated

`tests/regression/test_golden_ik.py` — fixed target→(reachable, collision_free) pairs. If IK output drifts on any push, the `regression` CI job goes red.

### 7.5 CI green on the sprint-2 merge PR

![CI green](./screenshots/ci_green.png)

*All 8 checks passed on PR #3 (post-sprint-2 hotfix): lint, unit, integration, regression — for both `pull_request` and `push` triggers.*

## 8. Repository layout

```
xlerobot/
├── services/
│   ├── nl_command/       # FastAPI :8010
│   ├── motion_planner/   # FastAPI :8020
│   ├── dispatcher/       # FastAPI :8030 + ZMQ PUSH
│   ├── actuation/        # FastAPI :8040 + MQTT publisher
│   ├── telemetry/        # ZMQ SUB → Timescale + Redis daemon
│   └── shared/schemas.py # frozen pydantic v2 contract types v1.0
├── sim/extension.py      # host-only pointer stub
├── kit-app-template/     # Omniverse Kit app + digitaltwin.xlerobot_extension
├── infra/
│   ├── docker-compose.yml
│   ├── mosquitto/mosquitto.conf
│   ├── timescaledb/init.sql
│   ├── nginx/nginx.conf             # planner-lb round-robin
│   └── grafana/                     # provisioning + dashboard JSON
├── tests/{unit,integration,system,regression}
├── contracts/interface-contracts.md
├── docs/
├── .github/workflows/ci.yml         # lint → unit → integration → regression
├── PLAN.md, TASK_ALLOCATION.md, README.md
```

## 9. Rubric coverage checklist

| Rubric line | Where satisfied |
|-------------|-----------------|
| Well defined microservices, function of each | §1 (12-row service catalog) |
| Contract per pair (route, port, protocol, data, initiation, conclusion) | §2 (11-row contract table) + `contracts/interface-contracts.md` |
| Containers per microservice | `services/*/Dockerfile` + `infra/docker-compose.yml`; §3.1 shows all healthy |
| Successful deployment | §3.1 (`docker compose ps`) + §4 (live command flow) |
| Scaling of microservices | §5 + `docs/SCALING_PROOF.md` + `infra/nginx/nginx.conf` |
| Persistent data (historical) | §6.1 → §6.3 (Timescale row count before/after restart) |
| Persistent state (latest) | §6.1 → §6.3 (Redis `state:latest` byte-identical) |
| Complete test suite (unit, interface, system) | §7 (unit + integration + system + regression) |

**All 8 rubric lines addressed with code and screenshot evidence.**